# FAIR²-FoodHealth Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the **FAIR²-FoodHealth: A Provenance-Rich Dataset and Knowledge Graph for Food Composition, Semantic Interoperability and Cardiometabolic Guidance** dataset using the `mlcroissant` library.

The workflow follows a Croissant-first approach:
- load the dataset from its Croissant JSON-LD URL,
- inspect dataset metadata,
- discover available record sets and fields by their `@id`,
- load record sets into pandas DataFrames,
- apply basic data processing and exploratory analysis,
- create a few practical visualizations.

### Dataset Source
Croissant schema URL:
`https://sen.science/doi/10.71728/senscience.3yz2-mb2d/fair2.json`

### Important note
This dataset description indicates that the package may contain a mix of source-derived empirical records, curated mappings, governance/provenance tables, and synthetic workflow examples. When analyzing loaded data, always inspect review and interpretation fields carefully and do not treat all tables as equivalent observational evidence.

In [ ]:
# Ensure required libraries are installed
!pip install -q mlcroissant pandas matplotlib seaborn

## 1. Data Loading
Load the Croissant metadata and initialize the dataset object. We also print a compact metadata summary.

As requested, metadata is accessed as a single object and not treated like a dictionary through subscripting.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 120)

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.3yz2-mb2d/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Dataset loaded successfully")
print(f"Name: {getattr(metadata, 'name', 'N/A')}")
print(f"Description: {getattr(metadata, 'description', 'N/A')}")
print(f"Version: {getattr(metadata, 'version', 'N/A')}")
print(f"License: {getattr(metadata, 'license', 'N/A')}")
print(f"Citation: {getattr(metadata, 'cite_as', getattr(metadata, 'citeAs', 'N/A'))}")

### Dataset metadata as JSON
For deeper inspection, we convert the metadata object to JSON-compatible form and review the top-level keys.

In [ ]:
metadata_json = metadata.to_json()
print("Top-level metadata keys:")
print(sorted(metadata_json.keys()))

## 2. Data Overview
This section discovers the available Croissant entities and emphasizes the use of `@id` values for all references.

Because Croissant datasets can vary in structure, the cells below are written defensively to inspect what is present before attempting extraction.

In [ ]:
# Inspect the serialized metadata to find record sets and related objects.
record_sets_json = metadata_json.get("recordSet", [])

print(f"Number of record sets described in metadata JSON: {len(record_sets_json)}")
if len(record_sets_json) == 0:
    print("No record sets were found in the top-level 'recordSet' field of the serialized metadata.")
    print("This can happen if the remote Croissant package is distribution-focused or if record sets are resolved differently by the library/runtime.")
else:
    for i, rs in enumerate(record_sets_json, start=1):
        rs_id = rs.get("@id", "N/A") if isinstance(rs, dict) else "N/A"
        rs_name = rs.get("name", "N/A") if isinstance(rs, dict) else "N/A"
        print(f"{i}. @id={rs_id} | name={rs_name}")

### Helper functions for overview
These helper functions flatten record-set and field information into simple tables. All references use `@id` when available.

In [ ]:
def field_rows_from_recordset(record_set_obj):
    rows = []
    fields = record_set_obj.get("field", []) if isinstance(record_set_obj, dict) else []
    for field in fields:
        if isinstance(field, dict):
            rows.append({
                "record_set_id": record_set_obj.get("@id", None),
                "record_set_name": record_set_obj.get("name", None),
                "field_id": field.get("@id", None),
                "field_name": field.get("name", None),
                "description": field.get("description", None),
                "data_type": field.get("dataType", None),
                "source": field.get("source", None),
            })
    return rows


record_set_rows = []
field_rows = []

for rs in record_sets_json:
    if isinstance(rs, dict):
        record_set_rows.append({
            "record_set_id": rs.get("@id", None),
            "record_set_name": rs.get("name", None),
            "description": rs.get("description", None),
            "field_count": len(rs.get("field", [])) if isinstance(rs.get("field", []), list) else 0,
        })
        field_rows.extend(field_rows_from_recordset(rs))

record_sets_df = pd.DataFrame(record_set_rows)
fields_df = pd.DataFrame(field_rows)

print("Record sets overview:")
display(record_sets_df.head(20))

print("Fields overview:")
display(fields_df.head(50))

### Preview records from a selected record set
The next cell demonstrates how to preview records from a record set using its `@id`.

If record sets are available, the notebook automatically selects the first discovered `@id`. Otherwise, it prints a helpful message.

In [ ]:
selected_record_set_id = None
if not record_sets_df.empty and "record_set_id" in record_sets_df.columns:
    selected_record_set_id = record_sets_df["record_set_id"].dropna().iloc[0] if not record_sets_df["record_set_id"].dropna().empty else None

print(f"Selected record set @id: {selected_record_set_id}")

if selected_record_set_id is not None:
    try:
        preview_records = []
        for idx, x in enumerate(dataset.records(record_set=selected_record_set_id)):
            preview_records.append(x)
            print(x)
            if idx >= 2:
                break
        if len(preview_records) == 0:
            print("No records were yielded for this record set.")
    except Exception as e:
        print(f"Could not iterate over records for record set {selected_record_set_id}: {e}")
else:
    print("No record set @id is available to preview from the metadata inspection.")

## 3. Data Extraction
Load data from one or more record sets into pandas DataFrames.

To stay faithful to Croissant best practice, record sets are referenced by their `@id` values. The code below dynamically discovers available record sets, loads them, and stores them in a dictionary keyed by record-set `@id`.

In [ ]:
# Build the list of record set @ids dynamically
record_set_ids = []
if not record_sets_df.empty and "record_set_id" in record_sets_df.columns:
    record_set_ids = [x for x in record_sets_df["record_set_id"].dropna().tolist() if x]

print("Discovered record set @ids:")
print(record_set_ids if record_set_ids else "None found")

dataframes = {}
load_errors = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded {len(dataframes[record_set_id])} rows from record set {record_set_id}")
    except Exception as e:
        load_errors[record_set_id] = str(e)
        print(f"Failed to load record set {record_set_id}: {e}")

print(f"\nSuccessfully loaded {len(dataframes)} DataFrame(s)")
if load_errors:
    print("Load errors:")
    for k, v in load_errors.items():
        print(f"- {k}: {v}")

In [ ]:
# Summarize loaded DataFrames
loaded_summary_rows = []
for record_set_id, df in dataframes.items():
    loaded_summary_rows.append({
        "record_set_id": record_set_id,
        "rows": len(df),
        "columns": len(df.columns),
        "column_names": list(df.columns),
    })

loaded_summary_df = pd.DataFrame(loaded_summary_rows)
display(loaded_summary_df)

### Inspect a loaded DataFrame
We select one available DataFrame and inspect its columns and first few rows. This helps identify which field `@id`s correspond to useful numeric or grouping variables.

In [ ]:
selected_loaded_record_set_id = next(iter(dataframes.keys()), None)
print(f"Selected loaded record set @id: {selected_loaded_record_set_id}")

if selected_loaded_record_set_id is not None:
    selected_df = dataframes[selected_loaded_record_set_id]
    print("Columns:")
    print(selected_df.columns.tolist())
    display(selected_df.head())
else:
    print("No DataFrame is available for inspection.")

## 4. Exploratory Data Analysis (EDA)
This section applies general-purpose EDA steps that adapt to the loaded tables:
- identify numeric columns,
- filter on a numeric field,
- normalize the selected field,
- optionally group by a categorical field,
- inspect missingness.

Because Croissant datasets may expose many heterogeneous tables, the logic below automatically chooses a reasonable numeric column from the selected record set.

In [ ]:
if selected_loaded_record_set_id is not None:
    eda_df = dataframes[selected_loaded_record_set_id].copy()
    
    # Convert obvious numeric-looking columns where possible
    for col in eda_df.columns:
        eda_df[col] = pd.to_numeric(eda_df[col], errors="ignore")
    
    numeric_columns = eda_df.select_dtypes(include="number").columns.tolist()
    object_columns = eda_df.select_dtypes(include="object").columns.tolist()
    
    print(f"Numeric columns in {selected_loaded_record_set_id}: {numeric_columns}")
    print(f"Object/categorical-like columns in {selected_loaded_record_set_id}: {object_columns[:20]}")
else:
    eda_df = pd.DataFrame()
    numeric_columns = []
    object_columns = []
    print("No DataFrame is available for EDA.")

In [ ]:
# Select a numeric field dynamically
numeric_field_id = numeric_columns[0] if numeric_columns else None
group_field_id = object_columns[0] if object_columns else None

print(f"Selected numeric field @id: {numeric_field_id}")
print(f"Selected grouping field @id: {group_field_id}")

if numeric_field_id is not None and not eda_df.empty:
    numeric_series = pd.to_numeric(eda_df[numeric_field_id], errors="coerce").dropna()
    
    if len(numeric_series) > 0:
        threshold = numeric_series.quantile(0.5)
        filtered_df = eda_df[pd.to_numeric(eda_df[numeric_field_id], errors="coerce") > threshold].copy()
        
        print(f"Using threshold based on median of {numeric_field_id}: {threshold}")
        print(f"Filtered rows: {len(filtered_df)}")
        display(filtered_df.head())
        
        std_value = pd.to_numeric(filtered_df[numeric_field_id], errors="coerce").std()
        mean_value = pd.to_numeric(filtered_df[numeric_field_id], errors="coerce").mean()
        
        if pd.notna(std_value) and std_value != 0:
            filtered_df[f"{numeric_field_id}_normalized"] = (
                pd.to_numeric(filtered_df[numeric_field_id], errors="coerce") - mean_value
            ) / std_value
        else:
            filtered_df[f"{numeric_field_id}_normalized"] = 0.0
        
        print("Normalized numeric field:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        
        if group_field_id is not None and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].agg(["count", "mean", "median", "min", "max"]).reset_index()
            print(f"Grouped summary by {group_field_id}:")
            display(grouped_df.head(20))
        else:
            grouped_df = pd.DataFrame()
            print("No grouping field available for grouped summary.")
    else:
        filtered_df = pd.DataFrame()
        grouped_df = pd.DataFrame()
        print("No non-null numeric values available for filtering.")
else:
    filtered_df = pd.DataFrame()
    grouped_df = pd.DataFrame()
    print("EDA could not proceed because no numeric field was detected.")

In [ ]:
# Missingness summary for the selected record set
if not eda_df.empty:
    missingness_df = pd.DataFrame({
        "field_id": eda_df.columns,
        "missing_count": eda_df.isna().sum().values,
        "missing_pct": (eda_df.isna().mean().values * 100).round(2),
    }).sort_values("missing_pct", ascending=False)
    
    print("Missingness summary:")
    display(missingness_df.head(20))
else:
    print("No selected DataFrame available for missingness analysis.")

### Optional: simple outlier trimming example
A common preprocessing step is to remove extreme values before plotting or downstream analysis. Here we use the IQR rule on the selected numeric field.

In [ ]:
if numeric_field_id is not None and not eda_df.empty:
    numeric_values = pd.to_numeric(eda_df[numeric_field_id], errors="coerce").dropna()
    if len(numeric_values) >= 4:
        q1 = numeric_values.quantile(0.25)
        q3 = numeric_values.quantile(0.75)
        iqr = q3 - q1
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr
        
        trimmed_df = eda_df[
            pd.to_numeric(eda_df[numeric_field_id], errors="coerce").between(lower_bound, upper_bound, inclusive="both")
        ].copy()
        
        print(f"Outlier trimming on field {numeric_field_id}")
        print(f"Lower bound: {lower_bound}")
        print(f"Upper bound: {upper_bound}")
        print(f"Original rows: {len(eda_df)} | Trimmed rows: {len(trimmed_df)}")
    else:
        trimmed_df = eda_df.copy()
        print("Not enough numeric values to compute IQR-based outlier trimming.")
else:
    trimmed_df = pd.DataFrame()
    print("No numeric field available for outlier trimming.")

## 5. Visualization
Visualize the selected numeric field and, when possible, show grouped summaries for a categorical field.

These charts are generic so they can adapt to whichever record set is successfully loaded.

In [ ]:
if numeric_field_id is not None and not eda_df.empty:
    plt.figure(figsize=(10, 5))
    sns.histplot(pd.to_numeric(eda_df[numeric_field_id], errors="coerce").dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field_id}\nRecord set: {selected_loaded_record_set_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()
else:
    print("Skipping histogram because no numeric field is available.")

In [ ]:
if numeric_field_id is not None and not eda_df.empty:
    plt.figure(figsize=(10, 4))
    sns.boxplot(x=pd.to_numeric(eda_df[numeric_field_id], errors="coerce").dropna())
    plt.title(f"Boxplot of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.tight_layout()
    plt.show()
else:
    print("Skipping boxplot because no numeric field is available.")

In [ ]:
if not grouped_df.empty and group_field_id is not None and numeric_field_id is not None:
    plot_df = grouped_df.sort_values("count", ascending=False).head(10)
    
    plt.figure(figsize=(12, 6))
    sns.barplot(data=plot_df, x=group_field_id, y="mean")
    plt.title(f"Mean {numeric_field_id} by {group_field_id} (top groups)")
    plt.xlabel(group_field_id)
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("Skipping grouped bar chart because grouped data is not available.")

## 6. Additional Metadata Context
This dataset package includes rich documentation about provenance, limitations, and intended use. The cell below extracts a few important contextual fields from the metadata JSON to support responsible analysis.

In [ ]:
context_fields = [
    "keywords",
    "dataUseCases",
    "dataLimitations",
    "dataBiases",
    "dataCollectionType",
]

for field_name in context_fields:
    print(f"\n{field_name}:")
    value = metadata_json.get(field_name, None)
    if value is None:
        print("  Not available")
    elif isinstance(value, list):
        for item in value:
            print(f"  - {item}")
    else:
        print(value)

## 7. Conclusion
In this notebook, we:
- loaded the FAIR²-FoodHealth dataset from its Croissant schema URL,
- inspected metadata through the `mlcroissant` dataset object,
- attempted to discover record sets and fields using their `@id` values,
- loaded available record sets into pandas DataFrames,
- ran flexible EDA steps that adapt to the structure of the loaded table,
- created basic visualizations for numeric and grouped analysis.

### Key takeaway
For Croissant-based datasets, the most robust workflow is:
1. inspect metadata first,
2. identify record sets and fields by `@id`,
3. load data dynamically,
4. validate which tables are empirical, synthetic, curated, or provenance-oriented before interpretation.

### Next steps
You can extend this notebook by:
- selecting specific record-set `@id`s once they are confirmed,
- joining related tables via identifier fields,
- filtering on provenance or review-state columns,
- building domain-specific analyses for foods, nutrients, mappings, hazards, or condition links.